# Preprocessing & EDA

Reproducible inventory and quality checks for the solar-cell image datasets.

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image, UnidentifiedImageError
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASETS = {'custom': ROOT / 'data/custom/perovskite_tandem', 'processed': ROOT / 'data/processed/dataset_split'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


In [ ]:
def build_manifest(root, dataset_name):
    rows = []
    for path in sorted(root.rglob('*')):
        if path.suffix.lower() not in IMAGE_EXTENSIONS or '.ipynb_checkpoints' in path.parts:
            continue
        relative = path.relative_to(root).parts
        split = relative[0] if len(relative) > 1 else 'unspecified'
        class_name = relative[1] if len(relative) > 2 else 'unspecified'
        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                rows.append({'dataset': dataset_name, 'path': str(path), 'split': split, 'class': class_name, 'width': image.width, 'height': image.height, 'mode': image.mode})
        except (OSError, UnidentifiedImageError):
            pass
    return pd.DataFrame(rows)

manifest = pd.concat([build_manifest(path, name) for name, path in DATASETS.items()], ignore_index=True)
manifest.groupby(['dataset', 'split', 'class']).size().rename('images').reset_index()


In [ ]:
custom = manifest[manifest.dataset == 'custom']
custom.groupby(['split', 'class']).size().unstack(fill_value=0).plot(kind='bar', figsize=(10, 4), title='Custom dataset distribution')
plt.ylabel('Images'); plt.tight_layout(); plt.show()


## Preprocessing checklist
- Validate image readability before training.
- Preserve train/validation/test boundaries.
- Convert to RGB and resize to 224 × 224.
- Match training-time pixel scaling at inference.
- Augment training data only.
- Record hashes and seeds for reproducibility.